In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [3]:
!pip install tilelang

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 47.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 87.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 58.5 MB/s eta 0:00:00


In [4]:
import tilelang
import tilelang.language as T
from tilelang import jit
import torch


def step_kernel(N, M, BLOCK_N, BLOCK_M, dtype, threads):


    @T.prim_func
    def main(A: T.Tensor((N, M), dtype), B: T.Tensor((N, M), dtype)): # A: input grid, B: output grid

        with T.Kernel(T.ceildiv(N, BLOCK_N), T.ceildiv(M, BLOCK_M), threads=threads) as (bn, bm):
            rowstart = bn * BLOCK_N
            colstart = bm * BLOCK_M
            for i, j in T.Parallel(BLOCK_N, BLOCK_M):
                x = rowstart + i
                y = colstart + j

                count: T.int32 = 0
                
                nx = T.clamp(x - 1, 0, N - 1)
                ny = T.clamp(y - 1, 0, M - 1)
                count += T.cast(A[nx, ny], T.int32)

                nx = T.clamp(x - 1, 0, N - 1)
                ny = T.clamp(y, 0, M - 1)
                count += T.cast(A[nx, ny], T.int32)

                nx = T.clamp(x - 1, 0, N - 1)
                ny = T.clamp(y + 1, 0, M - 1)
                count += T.cast(A[nx, ny], T.int32)

                nx = T.clamp(x, 0, N - 1)
                ny = T.clamp(y - 1, 0, M - 1)
                count += T.cast(A[nx, ny], T.int32)

                nx = T.clamp(x, 0, N - 1)
                ny = T.clamp(y + 1, 0, M - 1)
                count += T.cast(A[nx, ny], T.int32)

                nx = T.clamp(x + 1, 0, N - 1)
                ny = T.clamp(y - 1, 0, M - 1)
                count += T.cast(A[nx, ny], T.int32)

                nx = T.clamp(x + 1, 0, N - 1)
                ny = T.clamp(y, 0, M - 1)
                count += T.cast(A[nx, ny], T.int32)

                nx = T.clamp(x + 1, 0, N - 1)
                ny = T.clamp(y + 1, 0, M - 1)
                count += T.cast(A[nx, ny], T.int32)
                                                                    
                cell = T.cast(A[x, y], dtype)
                alive = T.cast(
                    (count == 3) or (count == 2 and cell > 0),
                    dtype
                )
                B[x, y] = alive
    
    return main

In [15]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time, random, torch
from tqdm.auto import tqdm

def make_grid(random_fill=True):
    if random_fill:
        return torch.tensor([[int(random.random() < 0.3) for _ in range(COLS)] for _ in range(ROWS)])
    return torch.tensor([[0] * COLS for _ in range(ROWS)])


def step(grid):
    new = [[0] * COLS for _ in range(ROWS)]
    for r in range(ROWS):
        for c in range(COLS):
            live_neighbors = sum(
                grid[max(0, min(ROWS-1, r + dr))][max(0, min(COLS-1, c + dc))]
                for dr in (-1, 0, 1)
                for dc in (-1, 0, 1)
                if (dr, dc) != (0, 0)
            )
            if grid[r][c]:
                new[r][c] = live_neighbors in (2, 3)
            else:
                new[r][c] = live_neighbors == 3
    return new


CELL    = 8
COLS    = 10000
ROWS    = 10000
PADDING = 8   # gap between the two grids

WIDTH  = COLS * CELL * 2 + PADDING
HEIGHT = ROWS * CELL + 16   # 16px for labels

program = step_kernel(ROWS, COLS, BLOCK_N=64, BLOCK_M=64, dtype=T.float16, threads=256)
kernel = tilelang.compile(program, out_idx=-1, target="cuda")#, execution_backend="cython")

# 10000x10000 step_kernel(ROWS, COLS, BLOCK_N=16, BLOCK_M=16, dtype=T.float16, threads=256) kernel() 100 steps: 0.138861s  (1.389 ms/step)
# 10000x10000 step_kernel(ROWS, COLS, BLOCK_N=16, BLOCK_M=16, dtype=T.float16, threads=512) kernel() 100 steps: 0.316749s  (3.167 ms/step)
# 10000x10000 step_kernel(ROWS, COLS, BLOCK_N=16, BLOCK_M=16, dtype=T.float16, threads=128) kernel() 100 steps: 0.320178s  (3.202 ms/step)
# 10000x10000 step_kernel(ROWS, COLS, BLOCK_N=32, BLOCK_M=32, dtype=T.float16, threads=256) kernel() 100 steps: 0.330081s  (3.301 ms/step)
# 10000x10000 step_kernel(ROWS, COLS, BLOCK_N=64, BLOCK_M=64, dtype=T.float16, threads=256) kernel() 100 steps: 0.389210s  (3.892 ms/step)


a = make_grid()
n_steps = 100

# without
# start = time.time()
# bcpu = a
# for i in tqdm(range(n_steps)):
#     bcpu = step(bcpu)
# elapsed = time.time() - start
# print(f"step() {n_steps} steps: {elapsed:.6f}s  ({elapsed/n_steps*1000:.3f} ms/step)")

# with kernel
start = time.time()
b = a.clone().cuda().half()
for i in range(n_steps):
    b = kernel(b)
elapsed = time.time() - start


clear_output(wait=True)

# plt.imshow(a.cpu(), cmap="binary")
# plt.title(f"original")
# plt.axis("off")
# plt.show()

print(f"kernel() {n_steps} steps: {elapsed:.6f}s  ({elapsed/n_steps*1000:.3f} ms/step)")

# plt.imshow(bcpu, cmap="binary")
# plt.title(f"cpu Step {i}")
# plt.axis("off")
# plt.show()

# plt.imshow(b.cpu(), cmap="binary")
# plt.title(f"gpu Step {i}")
# plt.axis("off")
# plt.show()


kernel() 100 steps: 0.389210s  (3.892 ms/step)
